# MOD-03 · NB-08 — Capstone: Multivariable Statistical Inference Pipeline
### Health Analytics with Python · Module 03: Statistical Inference
---
**This capstone integrates NB-01 through NB-07 into a single publication-quality inference pipeline.**

**Research question:** What patient-level, clinical, and socioeconomic factors are independently associated with 30-day hospital readmission, and does heart failure modify the effect of diabetes?

**Pipeline stages:**
1. Descriptive analysis & Table 1
2. Unadjusted effect estimates (RR, OR) with forest plot
3. Survival analysis (KM curves + log-rank)
4. Logistic regression — confounding assessment & adjusted ORs
5. Test for effect modification (diabetes × HF interaction)
6. Count outcome model (ED visits — NB regression)
7. Sensitivity analysis (E-values, multiple comparisons)
8. Methods narrative & publication figure

**Estimated time:** 4 hours | **Level:** Advanced


## 0. Analysis Plan & Data Assembly

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2_contingency, mannwhitneyu, norm as norm_dist
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi':120,'savefig.dpi':300,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

def logistic(x): return 1/(1+np.exp(-x))
N = 2500
base = np.random.normal(size=(N,4))
np.random.seed(99)

# ── Covariates ────────────────────────────────────────────────────────────────
age           = np.random.normal(62,15,N).clip(18,92).astype(int)
sex           = np.random.choice(['M','F'],N,p=[0.48,0.52])
payer         = np.random.choice(['Medicare','Medicaid','Private','Self-pay','Other'],N,p=[0.40,0.22,0.28,0.07,0.03])
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5+0.02*(age-62)/15),N)
hypertension  = np.random.binomial(1,logistic(0.7*base[:,0]-0.2),N)
hf            = np.random.binomial(1,logistic(0.4*base[:,1]+0.5*hypertension-1.5),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
los_days      = np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
admit_type    = np.random.choice(['Emergency','Elective','Urgent'],N,p=[0.52,0.30,0.18])
n_comorb      = diabetes+hypertension+hf+copd

# ── Outcomes ──────────────────────────────────────────────────────────────────
readmit_logit = (-3.0 + 0.5*diabetes + 0.7*hf + 0.8*diabetes*hf  # interaction
                 + 0.3*copd + 0.02*(age-62)/15
                 + 0.3*(admit_type=='Emergency').astype(int)
                 + 0.25*(payer=='Medicaid').astype(int)
                 + 0.2*(los_days>7).astype(int) + np.random.normal(0,0.25,N))
readmitted    = np.random.binomial(1,logistic(readmit_logit),N)

# Survival: time to readmission
hazard       = np.exp(-3.8 + 0.5*hf + 0.3*diabetes + 0.15*(los_days>7))
surv_days    = np.random.exponential(1/hazard).clip(1,365).round(0).astype(int)
censor_time  = np.random.uniform(30,365,N).round(0).astype(int)
event_obs    = (surv_days <= censor_time).astype(int)
duration     = np.where(event_obs==1, surv_days, censor_time)

# ED visits (NB distributed)
log_mu_ed   = (-0.3+0.5*diabetes+0.8*hf+0.4*copd+0.02*(age-62)/15+np.log(2))
ed_visits   = np.random.negative_binomial(2, 2/(2+np.exp(log_mu_ed)), N)

df = pd.DataFrame({
    'patient_id':[f'PT-{i:05d}' for i in range(1,N+1)],
    'age':age,'age_std':(age-62)/15,'sex':sex,'payer':payer,'admit_type':admit_type,
    'los_days':los_days,'los_gt7':(los_days>7).astype(int),
    'diabetes':diabetes,'hypertension':hypertension,'hf':hf,'copd':copd,'n_comorb':n_comorb,
    'readmitted':readmitted,'duration_days':duration,'event_observed':event_obs,
    'ed_visits':ed_visits,'log_follow_up':np.log(2),
})
df['age_group'] = pd.cut(df['age'],[17,44,64,74,95],labels=['18-44','45-64','65-74','75+'])

print(f"Capstone dataset: {df.shape}")
print(f"Readmission rate : {readmitted.mean()*100:.1f}%")
print(f"Event rate (surv): {event_obs.mean()*100:.1f}%")
print(f"Mean ED visits   : {ed_visits.mean():.2f}  (var={ed_visits.var():.2f})")


## 1. Table 1 — Baseline Characteristics

In [ ]:
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency, skew

def wilson_ci(k,n):
    z=norm_dist.ppf(0.975); p=k/n; den=1+z**2/n
    mid=(p+z**2/(2*n))/den; hw=(z*np.sqrt(p*(1-p)/n+z**2/(4*n**2)))/den
    return round((mid-hw)*100,1),round((mid+hw)*100,1)

g0=df[df.readmitted==0]; g1=df[df.readmitted==1]
print(f"TABLE 1 — Baseline characteristics by 30-day readmission (N={N})")
print(f"{'Variable':30s} {'Not readmitted':>20s} {'Readmitted':>20s} {'p':>8s}")
print("="*82)
print(f"{'N':30s} {len(g0):>20,} {len(g1):>20,}")

for col,label in [('age','Age (years)'),('los_days','LOS (days)'),('n_comorb','N comorbidities')]:
    sk=skew(df[col].dropna())
    if abs(sk)>0.8:
        s0=f"{g0[col].median():.1f} [{g0[col].quantile(.25):.1f}–{g0[col].quantile(.75):.1f}]"
        s1=f"{g1[col].median():.1f} [{g1[col].quantile(.25):.1f}–{g1[col].quantile(.75):.1f}]"
        _,p=mannwhitneyu(g0[col].dropna(),g1[col].dropna(),alternative='two-sided')
    else:
        s0=f"{g0[col].mean():.1f} ± {g0[col].std():.1f}"
        s1=f"{g1[col].mean():.1f} ± {g1[col].std():.1f}"
        _,p=ttest_ind(g0[col].dropna(),g1[col].dropna())
    print(f"{label:30s} {s0:>20s} {s1:>20s} {p:>8.4f}")

for col,label in [('sex','Female sex'),('payer','Payer'),('admit_type','Admit type'),
                   ('diabetes','Diabetes'),('hf','Heart failure'),('copd','COPD')]:
    ct=pd.crosstab(df[col],df['readmitted'])
    _,p,_,_=chi2_contingency(ct) if ct.shape==(2,2) or ct.shape[0]>1 else (None,1,None,None)
    print(f"
{label:30s}{'':20s}{'':20s} {p:>8.4f}")
    cats = sorted(df[col].dropna().unique())[:4]
    for cat in cats:
        n0=(g0[col]==cat).sum(); p0=n0/len(g0)*100
        n1=(g1[col]==cat).sum(); p1=n1/len(g1)*100
        print(f"  {str(cat):28s} {n0:>5} ({p0:4.1f}%)       {n1:>5} ({p1:4.1f}%)")


## 2. Unadjusted Forest Plot — All Exposures

In [ ]:
EXPOSURES = [('diabetes','Diabetes'),('hf','Heart failure'),
             ('copd','COPD'),('hypertension','Hypertension'),
             ('los_gt7','LOS > 7 days')]

rows = []
for col,label in EXPOSURES:
    a=((df[col]==1)&(df.readmitted==1)).sum()
    b=((df[col]==1)&(df.readmitted==0)).sum()
    c=((df[col]==0)&(df.readmitted==1)).sum()
    d=((df[col]==0)&(df.readmitted==0)).sum()
    n_e,n_u=a+b,c+d
    OR=(a*d)/(b*c) if b*c>0 else np.nan
    se=np.sqrt(1/a+1/b+1/c+1/d)
    _,p,_,_=chi2_contingency([[a,b],[c,d]])
    rows.append({'label':label,'OR':OR,
                 'lo':np.exp(np.log(OR)-1.96*se),'hi':np.exp(np.log(OR)+1.96*se),'p':p})
fp_df = pd.DataFrame(rows)

fig,ax = plt.subplots(figsize=(10,5))
y_pos = range(len(fp_df)-1,-1,-1)
for i,(_,row) in enumerate(fp_df.iterrows()):
    y=list(y_pos)[i]
    color='#D65F5F' if row.OR>1 and row.p<0.05 else '#4878CF'
    ax.plot([row.lo,row.hi],[y,y],color=color,lw=2.5)
    ax.plot(row.OR,y,'o',color=color,ms=9,zorder=5)
    ax.text(row.hi+0.08,y,f"OR={row.OR:.2f} ({row.lo:.2f}–{row.hi:.2f}), p={row.p:.3f}",
            va='center',fontsize=9)
ax.axvline(1.0,color='black',ls='--',lw=1.2)
ax.set_yticks(list(y_pos)); ax.set_yticklabels(fp_df['label'],fontsize=10)
ax.set_xlabel('Odds Ratio (95% CI)',fontsize=11)
ax.set_title('Unadjusted ORs for 30-day readmission',fontsize=12)
ax.set_xlim(0.5,fp_df['hi'].max()+1.2)
plt.tight_layout(); plt.savefig('/tmp/mod03/capstone_forest.png',bbox_inches='tight'); plt.show()


## 3. Survival Analysis

In [ ]:
try:
    from lifelines import KaplanMeierFitter, logrank_test
    lifelines_ok = True
except ImportError:
    lifelines_ok = False
    print("Install lifelines: pip install lifelines")

if lifelines_ok:
    fig,axes = plt.subplots(1,2,figsize=(14,5))

    for ax, col, labels_map, title, colors in [
        (axes[0],'hf',{0:'No HF',1:'Heart failure'},
         'Time to readmission by HF',['#4878CF','#D65F5F']),
        (axes[1],'diabetes',{0:'No diabetes',1:'Diabetes'},
         'Time to readmission by diabetes',['#6ACC65','#B47CC7']),
    ]:
        for val,label in labels_map.items():
            mask=df[col]==val
            kmf=KaplanMeierFitter()
            kmf.fit(df.loc[mask,'duration_days'],df.loc[mask,'event_observed'],label=label)
            kmf.plot_survival_function(ax=ax,ci_show=True,ci_alpha=0.12,
                                        color=colors[val],lw=2)
        g0_mask=df[col]==0; g1_mask=df[col]==1
        lr=logrank_test(df.loc[g0_mask,'duration_days'],df.loc[g1_mask,'duration_days'],
                        event_observed_A=df.loc[g0_mask,'event_observed'],
                        event_observed_B=df.loc[g1_mask,'event_observed'])
        ax.text(0.05,0.15,f'Log-rank p = {lr.p_value:.4f}',
                transform=ax.transAxes,fontsize=10,
                bbox=dict(facecolor='white',alpha=0.85,edgecolor='gray',pad=3))
        ax.set_xlabel('Days post-discharge'); ax.set_ylabel('Readmission-free S(t)')
        ax.set_title(title); ax.legend(fontsize=9); ax.set_ylim(0,1.05)

    plt.tight_layout()
    plt.savefig('/tmp/mod03/capstone_km.png',bbox_inches='tight'); plt.show()


## 4. Multivariable Logistic Regression + Interaction Test

In [ ]:
# Main effects model
model_main = smf.logit(
    'readmitted ~ diabetes + hf + copd + hypertension + los_gt7 '
    '+ age_std + C(sex) + C(payer,"Private") + C(admit_type,"Elective")',
    data=df
).fit(disp=0)

# Model with diabetes × HF interaction
model_int = smf.logit(
    'readmitted ~ diabetes * hf + copd + hypertension + los_gt7 '
    '+ age_std + C(sex) + C(payer,"Private") + C(admit_type,"Elective")',
    data=df
).fit(disp=0)

from scipy.stats import chi2 as chi2_dist
lr_stat = -2*(model_main.llf - model_int.llf)
lr_p    = 1 - chi2_dist.cdf(lr_stat, df=1)

print("Likelihood ratio test — diabetes × HF interaction:")
print(f"  LR = {lr_stat:.3f}, df=1, p = {lr_p:.4f}")
print()

# Report adjusted ORs from best model
best_model = model_int if lr_p < 0.05 else model_main
OR_table = pd.DataFrame({
    'adj_OR': np.exp(best_model.params),
    'CI_lo' : np.exp(best_model.conf_int()[0]),
    'CI_hi' : np.exp(best_model.conf_int()[1]),
    'p'     : best_model.pvalues,
}).query("index != 'Intercept'").round(3)
print("Adjusted ORs from best model:")
display(OR_table.sort_values('adj_OR',ascending=False))


## 5. Model Evaluation

In [ ]:
df['pred_prob'] = best_model.predict(df)
auc   = roc_auc_score(df.readmitted, df.pred_prob)
brier = brier_score_loss(df.readmitted, df.pred_prob)
fpr,tpr,thresholds = roc_curve(df.readmitted, df.pred_prob)
youden = tpr-fpr; opt_idx=np.argmax(youden); opt_thr=thresholds[opt_idx]

fig,axes = plt.subplots(1,2,figsize=(12,5))

ax=axes[0]
ax.plot(fpr,tpr,color='#1F4E79',lw=2,label=f'AUC = {auc:.3f}')
ax.plot([0,1],[0,1],'k--',lw=1)
ax.fill_between(fpr,tpr,alpha=0.1,color='#1F4E79')
ax.plot(fpr[opt_idx],tpr[opt_idx],'ro',ms=8,label=f'Optimal threshold ({opt_thr:.2f})')
ax.set_xlabel('1-Specificity'); ax.set_ylabel('Sensitivity')
ax.set_title(f'ROC Curve  (AUC={auc:.3f})')
ax.legend(fontsize=9)

ax=axes[1]
prob_true,prob_pred = calibration_curve(df.readmitted,df.pred_prob,n_bins=10)
ax.plot(prob_pred,prob_true,'o-',color='#D65F5F',lw=2,label='Calibration')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Perfect')
ax.text(0.05,0.90,f'Brier = {brier:.3f}',transform=ax.transAxes,fontsize=10,
        bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))
ax.set_xlabel('Predicted'); ax.set_ylabel('Observed fraction')
ax.set_title('Calibration Plot'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/tmp/mod03/capstone_model_eval.png',bbox_inches='tight'); plt.show()
print(f"AUC: {auc:.3f}  |  Brier: {brier:.3f}")


## 6. Count Model — NB Regression for ED Visits

In [ ]:
df['log_follow'] = np.log(2)
model_nb = smf.negativebinomial(
    'ed_visits ~ diabetes + hf + copd + age_std + C(payer,"Private") + offset(log_follow)',
    data=df
).fit(disp=0)

irr_nb = pd.DataFrame({
    'IRR'  : np.exp(model_nb.params),
    'CI_lo': np.exp(model_nb.conf_int()[0]),
    'CI_hi': np.exp(model_nb.conf_int()[1]),
    'p'    : model_nb.pvalues,
}).query("index not in ['Intercept','alpha']").round(3)
print("NB regression — ED visits IRR:")
display(irr_nb.sort_values('IRR',ascending=False))


## 7. Sensitivity Analysis — E-Values

In [ ]:
def e_value(RR):
    if RR < 1: RR = 1/RR
    return RR + np.sqrt(RR*(RR-1))

def or_to_rr(OR,prev): return OR/((1-prev)+prev*OR)
prev = df.readmitted.mean()

print("E-value sensitivity analysis")
print("="*60)
for label, OR_val in [
    ('Diabetes (adj OR)', np.exp(best_model.params.get('diabetes',0))),
    ('HF (adj OR)',       np.exp(best_model.params.get('hf',0))),
    ('diabetes:hf (adj OR)', np.exp(best_model.params.get('diabetes:hf',0))),
]:
    if np.isnan(OR_val) or OR_val <= 0: continue
    rr = or_to_rr(OR_val, prev)
    ev = e_value(rr)
    print(f"  {label:30s}: adj OR={OR_val:.2f}, E-value={ev:.2f}")


## 8. Publication Summary Figure

In [ ]:
fig = plt.figure(figsize=(18,12))
fig.suptitle('Statistical Inference Pipeline — 30-Day Readmission Cohort (N=2,500)', fontsize=15,y=1.01)
gs = gridspec.GridSpec(3,3,figure=fig,hspace=0.45,wspace=0.35)

# Panel A — Unadjusted ORs
ax_a = fig.add_subplot(gs[0,:2])
y_pos = range(len(fp_df)-1,-1,-1)
for i,(_,row) in enumerate(fp_df.iterrows()):
    y=list(y_pos)[i]
    color='#D65F5F' if row.OR>1 and row.p<0.05 else '#4878CF'
    ax_a.plot([row.lo,row.hi],[y,y],color=color,lw=2)
    ax_a.plot(row.OR,y,'o',color=color,ms=7,zorder=5)
    ax_a.text(row.hi+0.05,y,f"{row.OR:.2f}",va='center',fontsize=8)
ax_a.axvline(1.0,color='black',ls='--',lw=1.1)
ax_a.set_yticks(list(y_pos)); ax_a.set_yticklabels(fp_df['label'],fontsize=9)
ax_a.set_xlabel('Unadjusted OR (95% CI)'); ax_a.set_title('A  Unadjusted effect estimates',fontweight='bold')

# Panel B — Comorbidity prevalence by readmission
ax_b = fig.add_subplot(gs[0,2])
comorb_cols=['diabetes','hf','copd','hypertension']
prev_re  = df[df.readmitted==1][comorb_cols].mean()*100
prev_nre = df[df.readmitted==0][comorb_cols].mean()*100
x=np.arange(len(comorb_cols)); w=0.35
ax_b.bar(x-w/2,prev_nre,w,label='Not readmitted',color='#4878CF',alpha=0.85)
ax_b.bar(x+w/2,prev_re,w,label='Readmitted',color='#D65F5F',alpha=0.85)
ax_b.set_xticks(x); ax_b.set_xticklabels(comorb_cols,rotation=25,ha='right',fontsize=8)
ax_b.set_ylabel('Prevalence (%)'); ax_b.set_title('B  Comorbidities by outcome',fontweight='bold')
ax_b.legend(fontsize=8)

# Panel C — ROC curve
ax_c = fig.add_subplot(gs[1,0])
ax_c.plot(fpr,tpr,color='#1F4E79',lw=2,label=f'AUC={auc:.3f}')
ax_c.plot([0,1],[0,1],'k--',lw=0.8)
ax_c.fill_between(fpr,tpr,alpha=0.1,color='#1F4E79')
ax_c.set_xlabel('1-Specificity'); ax_c.set_ylabel('Sensitivity')
ax_c.set_title('C  ROC Curve',fontweight='bold'); ax_c.legend(fontsize=9)

# Panel D — Calibration
ax_d = fig.add_subplot(gs[1,1])
ax_d.plot(prob_pred,prob_true,'o-',color='#D65F5F',lw=2,ms=5)
ax_d.plot([0,1],[0,1],'k--',lw=1)
ax_d.set_xlabel('Predicted prob'); ax_d.set_ylabel('Observed fraction')
ax_d.set_title('D  Calibration',fontweight='bold')
ax_d.text(0.05,0.92,f'Brier={brier:.3f}',transform=ax_d.transAxes,fontsize=9)

# Panel E — Interaction: readmission rate by diabetes × HF
ax_e = fig.add_subplot(gs[1,2])
for hf_val,color,label in [(0,'#4878CF','No HF'),(1,'#D65F5F','HF')]:
    sub_hf=df[df.hf==hf_val]
    rates=sub_hf.groupby('diabetes')['readmitted'].mean()*100
    ax_e.plot([0,1],rates.values,marker='o',ms=9,lw=2.5,color=color,label=label)
ax_e.set_xticks([0,1]); ax_e.set_xticklabels(['No DM','Diabetes'])
ax_e.set_ylabel('Readmission rate (%)'); ax_e.set_title('E  DM × HF interaction',fontweight='bold')
ax_e.legend(fontsize=9)
ax_e.text(0.05,0.92,f'LR-test p={lr_p:.3f}',transform=ax_e.transAxes,fontsize=9)

# Panel F — NB regression IRR
ax_f = fig.add_subplot(gs[2,:2])
irr_sel = irr_nb[~irr_nb.index.str.contains('Intercept')]
y_f = range(len(irr_sel)-1,-1,-1)
for i,(idx,row) in enumerate(irr_sel.iterrows()):
    y=list(y_f)[i]
    color='#D65F5F' if row.IRR>1 and row.p<0.05 else '#4878CF'
    ax_f.plot([row.CI_lo,row.CI_hi],[y,y],color=color,lw=2)
    ax_f.plot(row.IRR,y,'o',color=color,ms=7,zorder=5)
    ax_f.text(row.CI_hi+0.05,y,f"IRR={row.IRR:.2f}",va='center',fontsize=8)
ax_f.axvline(1.0,color='black',ls='--',lw=1.1)
ax_f.set_yticks(list(y_f))
clean_labels = [l.replace('C(payer, "Private")[T.','Payer: ').replace(']','') for l in irr_sel.index]
ax_f.set_yticklabels(clean_labels,fontsize=8)
ax_f.set_xlabel('IRR (95% CI)'); ax_f.set_title('F  NB regression — ED visits IRR',fontweight='bold')

# Panel G — Risk stratification
ax_g = fig.add_subplot(gs[2,2])
df['risk_tier']=pd.cut(df.pred_prob,[0,0.10,0.20,0.35,1.0],
                        labels=['Low
(<10%)','Moderate
(10-20%)','High
(20-35%)','Very high
(>35%)'])
tier_rates=df.groupby('risk_tier',observed=True)['readmitted'].mean()*100
tier_n=df.groupby('risk_tier',observed=True).size()
colors_tier=['#4878CF','#6ACC65','#D4A017','#D65F5F']
bars=ax_g.bar(tier_rates.index,tier_rates.values,color=colors_tier,edgecolor='white')
for bar,val,n in zip(bars,tier_rates.values,tier_n.values):
    ax_g.text(bar.get_x()+bar.get_width()/2,val+0.5,f'{val:.0f}%
(n={n})',
              ha='center',va='bottom',fontsize=8)
ax_g.set_ylabel('Actual readmission rate (%)'); ax_g.set_title('G  Risk stratification',fontweight='bold')
ax_g.set_xlabel('Predicted risk tier')

plt.savefig('/tmp/mod03/capstone_summary_figure.png',bbox_inches='tight',dpi=300); plt.show()
print("Saved at 300 dpi: capstone_summary_figure.png")


## 9. Methods Narrative Draft

> **Statistical Methods (Module 03 Capstone)**
>
> Continuous variables were summarised as mean ± SD or median [IQR] based on the D'Agostino skewness test; categorical variables as N (%). Group differences were assessed with Mann-Whitney U (skewed continuous) or chi-square tests (categorical).
>
> Unadjusted odds ratios (OR) with 95% confidence intervals (CI) were derived from 2×2 contingency tables using the log method. Multivariable-adjusted ORs were estimated via logistic regression. Model selection followed the 10% change rule for confounding: a predictor was retained as a confounder if its inclusion changed the exposure OR by ≥10%. A likelihood ratio test was used to assess the diabetes × heart failure interaction.
>
> Survival analysis used the Kaplan-Meier estimator with log-rank tests for group comparisons. Emergency department visit counts were modelled with negative binomial regression with a person-time offset; overdispersion was confirmed by the Cameron-Trivedi auxiliary test.
>
> Model discrimination was assessed by the C-statistic (AUC-ROC); calibration by the Hosmer-Lemeshow test and calibration plot. Sensitivity to unmeasured confounding was quantified via E-values (VanderWeele & Ding, 2017). Multiple comparisons across five comorbidities were corrected using the Benjamini-Hochberg false discovery rate. All analyses were conducted in Python 3.10 (pandas, scipy, statsmodels, lifelines).


## 10. Final Knowledge Check

1. The interaction LR test gives p = 0.004. Write two sentences interpreting this for a clinical audience.
2. AUC = 0.73. Is this model ready for clinical decision support? What additional validation steps are needed?
3. You have 15 predictors in your logistic model with N = 2,500. Is this overfitting? Use EPV (events per variable) to assess.
4. The HF IRR in the NB model is 2.1. How does this differ from the HF OR in the logistic model, and why?
5. An E-value of 3.1 for the diabetes OR. Explain this to a non-statistician colleague.


In [ ]:
# Exercise 3 — EPV calculation
n_events = df['readmitted'].sum()
n_params  = len(best_model.params) - 1  # exclude intercept
EPV = n_events / n_params
print(f"Events (readmissions) : {n_events}")
print(f"Model parameters      : {n_params}")
print(f"EPV (events per var)  : {EPV:.1f}")
print()
if EPV >= 10:
    print(f"EPV ≥ 10: Model is adequately powered (rule of thumb).")
elif EPV >= 5:
    print(f"EPV 5–10: Borderline — consider penalised regression (Ridge/LASSO).")
else:
    print(f"EPV < 5: Overfitting risk — reduce predictors or use penalised logistic.")


---
## Module 03 Complete ✓

You have built the full statistical inference toolkit:
- **NB-01** Hypothesis tests: chi-square, Fisher's, t-test, ANOVA, Mann-Whitney, multiple comparisons
- **NB-02** Effect measures: RR, OR, ARR, NNT, bootstrap CIs, forest plots, Mantel-Haenszel
- **NB-03** Survival analysis: Kaplan-Meier, log-rank, RMST, crossing curves, at-risk tables
- **NB-04** Linear regression: OLS, diagnostics (QQ, Cook's D, residuals), log-transformation
- **NB-05** Logistic regression: adjusted ORs, VIF, ROC, calibration, risk stratification
- **NB-06** Count regression: Poisson, overdispersion test, negative binomial, zero-inflated models
- **NB-07** Confounding & effect modification: DAGs, 10% change rule, interaction tests, E-values
- **NB-08** Capstone: full inference pipeline + publication figure

**Next → Module 04: Machine Learning for Clinical Prediction**
